In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# 🤖 03 - Model Training\n", "Train and save the matching model"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.insert(0, os.path.abspath('../..'))\n",
    "print('Setup complete ✅')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from ml.src.train import generate_training_data\n",
    "import pandas as pd\n",
    "\n",
    "df = generate_training_data()\n",
    "print(f'Training samples: {len(df)}')\n",
    "print(f'Positive: {df[\"label\"].sum()}')\n",
    "print(f'Negative: {len(df) - df[\"label\"].sum()}')\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "from app.ml.models.similarity import SimilarityModel\n",
    "\n",
    "sim_model = SimilarityModel()\n",
    "X, y = [], []\n",
    "\n",
    "for i, row in df.iterrows():\n",
    "    print(f'Processing {i+1}/{len(df)}...')\n",
    "    vec = sim_model.feature_vector(row['resume'], row['job'])\n",
    "    X.append(vec[0])\n",
    "    y.append(row['label'])\n",
    "\n",
    "X = np.array(X)\n",
    "y = np.array(y)\n",
    "print(f'\\nFeature matrix shape: {X.shape}')\n",
    "print(f'Feature names: semantic, skill_match, experience, word_overlap, resume_skills, job_skills, matched_skills')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.metrics import accuracy_score, classification_report\n",
    "from app.ml.models.classifier import MatchClassifier\n",
    "\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42\n",
    ")\n",
    "\n",
    "print(f'Train: {X_train.shape}, Test: {X_test.shape}')\n",
    "\n",
    "classifier = MatchClassifier(model_type='xgboost')\n",
    "classifier.train(X_train, y_train)\n",
    "\n",
    "y_pred = classifier.model.predict(X_test)\n",
    "acc = accuracy_score(y_test, y_pred)\n",
    "print(f'\\nAccuracy: {acc:.3f}')\n",
    "print(classification_report(y_test, y_pred, target_names=['No Match','Match'], zero_division=0))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "\n",
    "feature_names = [\n",
    "    'Semantic Score', 'Skill Match', 'Experience',\n",
    "    'Word Overlap', 'Resume Skills', 'Job Skills', 'Matched Skills'\n",
    "]\n",
    "importances = classifier.model.feature_importances_\n",
    "sorted_idx = np.argsort(importances)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(8, 5))\n",
    "ax.barh(\n",
    "    [feature_names[i] for i in sorted_idx],\n",
    "    [importances[i] for i in sorted_idx],\n",
    "    color='steelblue'\n",
    ")\n",
    "ax.set_title('Feature Importance')\n",
    "ax.set_xlabel('Importance Score')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "classifier.save()\n",
    "print('\\n✅ Model saved to app/ml/artifacts/model.pkl')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}